# Modèle Prédictif — Meta Score T+1 mois

**Objectif :** prédire le `meta_score` d'un archetype le mois prochain à partir de ses caractéristiques actuelles.

**Features :**
- Score et trend actuels (meta_score, share, avg_placement, trend_ratio)
- Structure du deck (avg_jaccard des paires core, densité du sous-graphe)
- Contexte banlist (nombre de cartes bannies / limitées dans l'archetype)
- Nombre de staples de format jouées

**Split temporel :** entraînement sur 2024-2025, test sur 2026 (évite la fuite de données).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

con = sqlite3.connect('../data/yugioh.db')

# ── Charger meta_scores ───────────────────────────────────────────────────────
ms = pd.read_sql("""
    SELECT month, archetype, deck_count, total_decks, share, avg_placement, meta_score
    FROM meta_scores
    ORDER BY month, archetype
""", con)
ms['month'] = pd.to_datetime(ms['month'])
print(f'meta_scores chargé : {len(ms):,} lignes')

## 1. Construction des features

In [ ]:
# ── Features depuis la DB ─────────────────────────────────────────────────────

# trend_ratio par archetype
trend = pd.read_sql("SELECT archetype, trend_ratio FROM archetype_trend", con)

# Cartes bannies / limitées par archetype
banlist = pd.read_sql("""
    SELECT td.archetype,
           SUM(CASE WHEN c.ban_tcg = 'Banned'       THEN 1 ELSE 0 END) AS n_banned,
           SUM(CASE WHEN c.ban_tcg = 'Limited'       THEN 1 ELSE 0 END) AS n_limited,
           SUM(CASE WHEN c.ban_tcg = 'Semi-Limited'  THEN 1 ELSE 0 END) AS n_semilimited
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    JOIN cards c ON c.name = dc.card_name
    WHERE td.illegal = 0 AND dc.zone = 'main' AND td.archetype IS NOT NULL
    GROUP BY td.archetype
""", con)

# Staples de format par archetype (cartes présentes dans >5 archetypes différents)
staples_query = pd.read_sql("""
    SELECT td.archetype,
           COUNT(DISTINCT dc.card_name) AS n_staples
    FROM deck_cards dc
    JOIN tournament_decks td ON td.id = dc.deck_id
    WHERE td.illegal = 0 AND dc.zone = 'main'
      AND dc.card_name IN (
          SELECT card_name FROM (
              SELECT dc2.card_name, COUNT(DISTINCT td2.archetype) AS n_arch
              FROM deck_cards dc2
              JOIN tournament_decks td2 ON td2.id = dc2.deck_id
              WHERE td2.illegal = 0 AND dc2.zone = 'main' AND td2.archetype IS NOT NULL
              GROUP BY dc2.card_name
              HAVING n_arch > 5
          )
      )
    GROUP BY td.archetype
""", con)

# Densité de co-occurrence par archetype (avg Jaccard des paires core)
cooc_features = pd.read_sql("""
    SELECT td.archetype,
           AVG(cc.jaccard)   AS avg_jaccard,
           COUNT(cc.jaccard) AS n_pairs,
           MAX(cc.jaccard)   AS max_jaccard
    FROM card_cooccurrence cc
    JOIN deck_cards dc1 ON dc1.card_name = cc.card_a
    JOIN tournament_decks td ON td.id = dc1.deck_id
    WHERE td.illegal = 0 AND td.archetype IS NOT NULL
    GROUP BY td.archetype
""", con)

con.close()

# Fusionner toutes les features
features = (trend
    .merge(banlist,        on='archetype', how='left')
    .merge(staples_query,  on='archetype', how='left')
    .merge(cooc_features,  on='archetype', how='left')
    .fillna(0))

print(f'Features calculées pour {len(features)} archetypes')
print(features.head())

## 2. Dataset d'entraînement — T → T+1 mois

In [ ]:
# Pour chaque (mois, archetype), créer un exemple :
#   X = features au mois T  +  meta_score à T
#   y = meta_score à T+1
ms_sorted = ms.sort_values(['archetype', 'month'])
ms_sorted['meta_score_next'] = ms_sorted.groupby('archetype')['meta_score'].shift(-1)
ms_sorted['month_next'] = ms_sorted.groupby('archetype')['month'].shift(-1)

# Garder seulement les lignes où T+1 existe ET est le mois suivant
ms_sorted = ms_sorted.dropna(subset=['meta_score_next'])
ms_sorted = ms_sorted[
    (ms_sorted['month_next'] - ms_sorted['month']).dt.days.between(25, 35)
]

# Fusionner avec les features statiques
dataset = ms_sorted.merge(features, on='archetype', how='left').fillna(0)

FEATURE_COLS = [
    'meta_score', 'share', 'avg_placement',
    'trend_ratio', 'n_banned', 'n_limited', 'n_semilimited',
    'n_staples', 'avg_jaccard', 'n_pairs', 'max_jaccard'
]

X = dataset[FEATURE_COLS].values
y = dataset['meta_score_next'].values

print(f'Dataset : {len(dataset)} exemples, {len(FEATURE_COLS)} features')
print(f'Target  : min={y.min():.3f}  max={y.max():.3f}  mean={y.mean():.3f}')

## 3. Split temporel + entraînement

In [ ]:
# Split temporel : train = avant 2026-01, test = 2026+
SPLIT_DATE = pd.Timestamp('2026-01-01')
train_mask = dataset['month'] < SPLIT_DATE
test_mask  = dataset['month'] >= SPLIT_DATE

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train : {len(X_train)} exemples ({dataset[train_mask]["month"].min().date()} → {dataset[train_mask]["month"].max().date()})')
print(f'Test  : {len(X_test)} exemples ({dataset[test_mask]["month"].min().date()} → {dataset[test_mask]["month"].max().date()})')
print()

# Normalisation
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# ── Modèles ───────────────────────────────────────────────────────────────────
models = {
    'Ridge Regression':    Ridge(alpha=1.0),
    'Random Forest':       RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}

results = {}
for name, model in models.items():
    if name == 'Ridge Regression':
        model.fit(X_train_s, y_train)
        preds = model.predict(X_test_s)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'model': model, 'preds': preds, 'rmse': rmse, 'r2': r2}
    print(f'{name:25s}  RMSE={rmse:.4f}  R²={r2:.3f}')

## 4. Feature importance (Random Forest)

In [ ]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)

print('Feature importance (Random Forest) :')
for feat, imp in importances.items():
    bar = '█' * int(imp * 40)
    print(f'  {feat:20s}  {imp:.3f}  {bar}')

## 5. Prédictions pour le mois prochain

In [ ]:
# Prendre le dernier mois connu pour chaque archetype
last_known = (ms.sort_values('month')
              .groupby('archetype')
              .last()
              .reset_index())

last_known = last_known.merge(features, on='archetype', how='left').fillna(0)

X_future = last_known[FEATURE_COLS].values
X_future_s = scaler.transform(X_future)

# Utiliser le meilleur modèle (Gradient Boosting généralement)
best_model_name = max(results, key=lambda k: results[k]['r2'])
best_model = results[best_model_name]['model']

if best_model_name == 'Ridge Regression':
    preds_future = best_model.predict(X_future_s)
else:
    preds_future = best_model.predict(X_future)

last_known['predicted_next_month'] = preds_future
last_known['delta'] = last_known['predicted_next_month'] - last_known['meta_score']

predictions = last_known[['archetype', 'meta_score', 'predicted_next_month', 'delta']].sort_values('predicted_next_month', ascending=False)

print(f'Prédictions mois prochain (modèle : {best_model_name}) :')
print()
print('=== ARCHETYPES QUI VONT MONTER ===')
print(predictions[predictions['delta'] > 0].head(10).to_string(index=False, float_format='{:.3f}'.format))
print()
print('=== ARCHETYPES QUI VONT BAISSER ===')
print(predictions[predictions['delta'] < 0].sort_values('delta').head(10).to_string(index=False, float_format='{:.3f}'.format))